In [2]:
import os
import pickle
import random
import torch
import numpy as np
import mediapipe as mp
import cv2

class MLP_hands(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.W1 = torch.nn.Parameter(torch.randn(63, 128) * 0.1)
        self.W2 = torch.nn.Parameter(torch.randn(128, 64) * 0.1)
        self.W3 = torch.nn.Parameter(torch.randn(64, 9) * 0.1)
        self.B1 = torch.nn.Parameter(torch.zeros(128))
        self.B2 = torch.nn.Parameter(torch.zeros(64))
        self.B3 = torch.nn.Parameter(torch.zeros(9))
    
    def forward(self, a1):
        z1 = a1 @ self.W1 + self.B1
        a2 = torch.relu(z1)
        z2 = a2 @ self.W2 + self.B2
        a3 = torch.relu(z2)
        ret = a3 @ self.W3 + self.B3
        return ret

model = MLP_hands()
model.load_state_dict(torch.load('working_model.pth'))

<All keys matched successfully>

In [ ]:
hands = mp.solutions.hands.Hands(static_image_mode=False)
cam = cv2.VideoCapture(0)

classes = ['1', '2', '3', '4', '5', '+', '-', '=', 'del']

ds = ""

while True:
    _, frame = cam.read()
    result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if cv2.waitKey(1) == ord('s') and result.multi_hand_landmarks:
        coords = []
        for p in result.multi_hand_landmarks[0].landmark:
            coords.append(p.x)
            coords.append(p.y)
            coords.append(p.z)
        batch = torch.tensor([coords], dtype=torch.float32)
        ret = model.forward(batch)
        imax = 0
        for i in range(9):
            if ret[0][i] > ret[0][imax]:
                imax = i
        if classes[imax] == "del":
            ds = ds[:-1]
        elif classes[imax] == "=":
            ans = 0
            op = 1
            now = ""
            for c in ds:
                if c == "+":
                    if now == "":
                        continue
                    ans += int(now) * op
                    op = 1
                    now = ""
                elif c == "-":
                    if now == "":
                        continue
                    ans += int(now) * op
                    op = -1
                    now = ""
                else:
                    now += c
            ans += int(now) * op
            ds = str(ans)
        else:
            ds += classes[imax]
            
    if result.multi_hand_landmarks and len(result.multi_hand_landmarks) == 1:
        cv2.circle(frame, (30, 30), 15, (0, 255, 0), -1)
    else:
        cv2.circle(frame, (30, 30), 15, (0, 0, 255), -1)
        
    h, w = frame.shape[:2]
    frame1 = np.zeros((h + 100, w, 3), dtype=np.uint8)
    frame1[:h, :] = frame
    cv2.putText(frame1, ds, (20, h + 60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)
    cv2.imshow("", frame1)
cam.release()
cv2.destroyAllWindows()
